# Cleaning: Part-1

The goal now is to post-process the job descriptions you collected in `HW-3.1-crawl.ipynb`.

In this section you MUST do it manually using traditional data science cleaning and parsing skills, i.e. without use of ANY LLM tools or wrappers. (You will do the LLM version in `HW-3.3-cleaning-2.ipynb`, and compare the two in your EDA.)

The unit of analysis in this case is "job", i.e. one row per unique job posting.

Your task is to write a cleaning script that parses the crawl results and creates a data frame, saved to `data/processed-jobs-1.csv`. This data frame will be used for later EDA.

## Getting started

Some tips for working with the raw crawl files:

* Load every JSON file in `data/` (e.g. with `glob`), and collect the entries of each file's `jobs_results` list. Skip the provided `example-out-...` file, or keep it, but make that choice deliberately.
* Each job has useful semi-structured fields in addition to the free-text `description`, e.g. `title`, `company_name`, `location`, `via` (the posting platform), `detected_extensions` (posting age, schedule type, salary, benefits flags), `job_highlights` (qualifications, responsibilities, benefits), and `job_id`. Field availability varies from job to job, so code defensively (e.g. `.get()`).
* `posted_at` values are relative (e.g. "4 days ago"). To convert them to dates, combine them with the time of the crawl, which is available in each file's `search_metadata` (or in your timestamped filename).
* The same job often appears under several search queries. `job_id`, or a combination of title, company, and location, is a good starting point for deduplication.
* Keep track of which search query each job came from; it is a useful feature in its own right.

## Features to extract

Try to extract as many features as possible. Here are some possible suggestions, which together provide a comprehensive overview for analyzing job postings for trends, patterns, and insights.

You will almost certainly not be able to extract all of these features, but do your best to make the most detailed data set that you can. You can also ADD more features to this list if they come to mind.

- **Job Title**
- **Company Name**
- **Sector/Industry**
- **Location** (City, State, Country)
- **Job Type** (Full-time, Part-time, Contract, Internship)
- **Salary** (Range or Exact)
- **Experience Level** (Entry-level, Mid-level, Senior-level)
- **Education Requirements** (Degree, Major, Certifications)
- **Skills/Technologies Required** (e.g., Python, SQL, Machine Learning, Cloud)
- **Job Responsibilities/Duties**
- **Required Years of Experience**
- **Benefits** (Health insurance, Retirement plans, Paid time off)
- **Remote Work Options** (Remote, Hybrid, On-site)
- **Application Deadline**
- **Job Posting Date**
- **Job Description Length** (Number of words or characters)
- **Keywords/Frequency of Terms**
- **Certifications Required or Preferred** (e.g., AWS Certified, PMP)
- **Team Size** (If mentioned)
- **Company Size** (Small, Medium, Large)
- **Company Reputation/Ranking** (If available)
- **Job Posting Platform** (Where the job was posted, e.g., LinkedIn, Indeed)
- **Company Values or Culture** (Diversity, Innovation, Sustainability)
- **Visa Sponsorship Availability**
- **Interview Process Information** (If mentioned)
- **Expected Start Date**
- **Job Posting Expiry Date**
- **Gender Diversity Language** (If any)
- **Working Hours/Shift Type** (Day shift, Night shift, Flexible hours)
- **Required Language Skills**
- **Job Location Proximity to Major Cities** (If provided)
- **Travel Requirements** (Percentage or Frequency)
- **Team Collaboration Tools Mentioned** (Slack, Zoom, etc.)
- **Reporting Line** (e.g., Reports to Senior Manager)
- **Job Benefits Related to Learning & Development** (e.g., Courses, Training)
- **Company Stock Options** (If offered)
- **Required Soft Skills** (e.g., Communication, Leadership)
- **Company Perks** (Gym membership, Free meals, Company car)
- **Job Posting ID or Reference Number**

## Cleaning checklist

While cleaning, try to address the following as best you can:

- **Missing Data**: Identify missing or null values in features like salary, company name, or location.
- **Duplicates**: Check for duplicate job listings or job descriptions.
- **Inconsistent Formatting**: Ensure consistency in categorical variables (e.g., job titles, location formats).
- **Data Type Validation**: Ensure each feature has the appropriate data type (e.g., salary as numeric, dates as datetime).
- **Outliers**: Detect outliers in salary, years of experience, or job description length.

In [62]:
# INSERT CODE HERE
import glob
import json
import os
import pandas as pd
import re

all_jobs = pd.DataFrame()

extract_columns = [
    "job_highlights",
    "extensions",
    "apply_options"
]

file_paths = sorted(
    glob.glob("data/*.json"),
    key=os.path.getmtime,
    reverse=True,
)[:90]

for file_path in file_paths:
    try:
        with open(file_path, encoding="utf-8") as f:
            saved = json.load(f)
    except (OSError, UnicodeDecodeError, json.JSONDecodeError) as e:
        print(f"Skipping {file_path}: {e}")
        continue

    jobs = saved.get("jobs_results", [])
    df = pd.json_normalize(saved["jobs_results"])

    for col in extract_columns:
        for row_index, value in df[col].items():
                if not isinstance(value, list):
                    continue
    
                if col == "job_highlights":
                    for section in value:
                        column_name = f"highlights_{section['title']}"

                        if column_name not in df.columns:
                            df[column_name] = ""

                        df.at[row_index, column_name] = "\n".join(
                            section.get("items", [])
                        )
                elif col == "apply_options":
                    for section in value:
                        column_name = f"apply_options_{section['title']}"

                        if column_name not in df.columns:
                            df[column_name] = ""

                        df.at[row_index, column_name] = section.get("link", "")

    df = df.drop(columns = ["job_highlights", "apply_options"], errors = "ignore")

    employment_types = {
            "full-time",
            "part-time",
            "contractor",
            "contract",
            "temporary",
            "internship",
        }

    boolean_columns = {
        "work from home": "work_from_home",
        "no degree mentioned": "no_degree_mentioned",
        "health insurance": "health_insurance",
        "dental insurance": "dental_insurance",
        "paid time off": "paid_time_off",
    }

    # Initialize text and Boolean columns.
    for column in [
        "posted_ago",
        "salary_text",
        "employment_type",
        "other_extensions",
    ]:
        df[column] = ""

    for column in boolean_columns.values():
        df[column] = False

    for row_index, values in df["extensions"].items():
        if not isinstance(values, list):
            continue

        other_extensions = []

        for extension in values:
            if not isinstance(extension, str):
                continue

            text = extension.strip()
            lower = text.lower()

            if (
                re.search(
                    r"\b\d+\+?\s+(?:minute|hour|day|week|month)s?\s+ago\b",
                    lower,
                )
                or lower in {"today", "yesterday"}
            ):
                df.at[row_index, "posted_ago"] = text

            elif (
                re.search(r"\d", text)
                and re.search(
                    r"\b(?:an?|per)\s+(?:hour|day|week|month|year)\b",
                    lower,
                )
            ):
                df.at[row_index, "salary_text"] = text

            elif lower in employment_types:
                df.at[row_index, "employment_type"] = text

            elif lower in boolean_columns:
                column_name = boolean_columns[lower]
                df.at[row_index, column_name] = True

            else:
                other_extensions.append(text)

        df.at[row_index, "other_extensions"] = " | ".join(
            other_extensions
        )

    all_jobs = pd.concat([all_jobs, df], ignore_index=True)

    print(f"Loaded {len(jobs)} jobs from {file_path}")

ages = (
    all_jobs["posted_ago"]
    .astype("string")
    .str.lower()
    .str.strip()
    .replace({"today": "0 days", "yesterday": "1 day"})
    .str.replace(r"\s+ago$", "", regex=True)
)

all_jobs["date_posted"] = (
    pd.Timestamp.now() - pd.to_timedelta(ages, errors="coerce")
).dt.normalize()


print(f"Total jobs loaded: {len(all_jobs)}")
all_jobs.to_csv("test.csv")

Loaded 7 jobs from data/Data-Architect-20261023-143824.json
Loaded 9 jobs from data/Data-Architect-20261023-143823.json
Loaded 10 jobs from data/Data-Architect-20261023-143820.json
Loaded 10 jobs from data/Risk-Analyst-20261023-143704.json
Loaded 8 jobs from data/Risk-Analyst-20261023-143702.json
Loaded 10 jobs from data/Risk-Analyst-20261023-143656.json
Loaded 7 jobs from data/Fraud-Detection-Specialist-20261023-143540.json
Loaded 10 jobs from data/Fraud-Detection-Specialist-20261023-143539.json
Loaded 10 jobs from data/Fraud-Detection-Specialist-20261023-143537.json
Loaded 10 jobs from data/Computational-Scientist-20261023-143421.json
Loaded 10 jobs from data/Computational-Scientist-20261023-143420.json
Loaded 10 jobs from data/Computational-Scientist-20261023-143418.json
Loaded 8 jobs from data/Cloud-AI-Engineer-20261023-143302.json
Loaded 10 jobs from data/Cloud-AI-Engineer-20261023-143301.json
Loaded 10 jobs from data/Cloud-AI-Engineer-20261023-143252.json
Loaded 10 jobs from data

In [97]:
## Cleaning checklist
# - **Data Type Validation**: Ensure each feature has the appropriate data type (e.g., salary as numeric, dates as datetime).
# - **Outliers**: Detect outliers in salary, years of experience, or job description length.

df = pd.read_csv("test.csv")

print(df.columns)

print(df.isna().sum())

cols = ["title", "company_name", "location", "job_id", "share_link"]  # Columns that should uniquely identify a row

duplicate_rows = (
    df[df.duplicated(subset=cols, keep=False)]
    .sort_values(cols)
)

df = df.drop_duplicates(subset = cols)

duplicate_rows = (
    df[df.duplicated(subset=cols, keep=False)]
    .sort_values(cols)
)

print(df.dtypes)

df["date_posted"] = pd.to_datetime(df["date_posted"])

location = df["location"].astype("string").str.strip()
parts = location.str.rsplit(",", n=1)

df["state"] = parts.str[-1].str.strip()
df["city"] = parts.str[0].where(
    location.str.contains(",", na=False)
).str.strip()

df = df.drop(columns = ["location", "posted_ago"])

na_counts = df.isna().sum()
na_counts[na_counts > 0].sort_values(ascending=False)

df = df.loc[:, df.isna().sum() < 500]

df.to_csv('data/processed-jobs-1.csv')

Index(['Unnamed: 0', 'title', 'company_name', 'location', 'via', 'share_link',
       'extensions', 'source_link', 'job_title', 'description',
       ...
       'apply_options_JustJobs.com', 'apply_options_Higher Ed Jobs',
       'apply_options_NukeWorker.com', 'apply_options_AI Search',
       'apply_options_Pitt Career Central - University Of Pittsburgh',
       'apply_options_Altrix', 'apply_options_LGBTQ Hire',
       'apply_options_The BIG Jobsite', 'apply_options_Hirefinity',
       'date_posted'],
      dtype='str', length=594)
Unnamed: 0                         0
title                              0
company_name                       0
location                           0
via                                0
                                ... 
apply_options_Altrix             831
apply_options_LGBTQ Hire         831
apply_options_The BIG Jobsite    831
apply_options_Hirefinity         831
date_posted                      327
Length: 594, dtype: int64
Unnamed: 0                

When you are done, save the result to `data/processed-jobs-1.csv`, and commit your work.